# Age stratification

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/07-age-stratification` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

summer4 has no `AgeStratification` class (ledger `S6` stays `partial`). Age
bands are an ordinary `Property`, population splits use `Split`, and ageing is
a `TraitChain` flow.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Split,
    TraitChain,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0", "20", "40", "60"))
pmap = PropertyMap.from_property(state).stratify(age)

model = FlowModel(pmap)
mixing = MixingMatrix(age, np.ones((4, 4)), check_reciprocal=False)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=mixing.prop,
            mixing=mixing,
            kind="frequency",
            contact_rate=1.0,
        ),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
ageing = TraitChain(
    age,
    (("0", "20"), ("20", "40"), ("40", "60")),
    rates=(1.0 / 20.0, 1.0 / 20.0, 1.0 / 20.0),
)
model.add_flow(
    TransitionFlow("ageing", age.present(), age.present(), 1.0, pairing=ageing)
)
model.set_initial_population(
    {state["S"]: 990.0, state["I"]: 10.0},
    splits=(Split(age, {"0": 0.25, "20": 0.25, "40": 0.25, "60": 0.25}),),
)
cm = model.compile()
y0 = cm.initial_state({})
assert np.isclose(float(np.asarray(y0.data).sum()), 1000.0)
for a in age.traits:
    assert np.isclose(float(np.asarray(y0.data)[pmap.select(age[a])].sum()), 250.0)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())})
res = cm.run({}, t0=0.0, t1=40.0, dt=0.5, save=plan, solver="euler")
frame = res["comp"].to_pandas()
assert frame.shape[0] > 5
frame.plot(title="Age-stratified SIR with ageing")
